In [ ]:
!pip install gradio pandas numpy matplotlib

In [ ]:
!pip install gradio

In [ ]:
!pip install fpdf

  Preparing metadata (setup.py) ... done
  Created wheel for fpdf: filename=fpdf-1.7.2-py2.py3-none-any.whl size=40704 sha256=bc9fc58b1508b9fd6c1822e81759782f4e9b01eb90acb52951ffcc90477539a3
  Stored in directory: /root/.cache/pip/wheels/6e/62/11/dc73d78e40a218ad52e7451f30166e94491be013a7850b5d75
Successfully built fpdf


In [ ]:
!mkdir -p assets

In [27]:
import gradio as gr
import pandas as pd
import numpy as np
from io import BytesIO
import matplotlib.pyplot as plt
import datetime
import os
import warnings
import time
warnings.filterwarnings('ignore')

# SQLite Database
from sqlalchemy import create_engine, Column, Integer, String, Float, DateTime, Boolean, Text
from sqlalchemy.ext.declarative import declarative_base
from sqlalchemy.orm import sessionmaker

# ML Models
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

try:
    from prophet import Prophet
    PROPHET_AVAILABLE = True
except ImportError:
    PROPHET_AVAILABLE = False
    print("⚠️ Prophet not installed. Using fallback forecasting.")

# ==================== DATABASE SETUP ====================

DATABASE_URL = "sqlite:///./msme_data.db"
engine = create_engine(DATABASE_URL, connect_args={"check_same_thread": False})
SessionLocal = sessionmaker(autocommit=False, autoflush=False, bind=engine)
Base = declarative_base()

class MSMEProfile(Base):
    __tablename__ = "msme_profiles"

    id = Column(Integer, primary_key=True, index=True)
    mobile_number = Column(String(15), unique=True, index=True)
    full_name = Column(String(100))
    email = Column(String(100))
    role = Column(String(50))
    company_name = Column(String(200))
    business_type = Column(String(50))
    state = Column(String(50))
    city = Column(String(100))
    years_operation = Column(Integer)
    monthly_revenue_range = Column(String(50))
    verification_status = Column(String(20), default="PENDING")
    created_at = Column(DateTime, default=datetime.datetime.utcnow)
    consent_given = Column(Boolean, default=False)
    organisation_type = Column(String(100))
    major_activity = Column(String(200))
    enterprise_type = Column(String(50))

Base.metadata.create_all(bind=engine)

# ==================== DATABASE OPERATIONS ====================

def save_user_profile(profile_data):
    """Save user profile to database"""
    db = SessionLocal()
    try:
        existing = db.query(MSMEProfile).filter(
            MSMEProfile.mobile_number == profile_data['mobile_number']
        ).first()

        profile_data_for_db = profile_data.copy()
        if 'msme_number' in profile_data_for_db:
            del profile_data_for_db['msme_number']

        if existing:
            for key, value in profile_data_for_db.items():
                if hasattr(existing, key):
                    setattr(existing, key, value)
            db.commit()
            return existing.id
        else:
            profile = MSMEProfile(**profile_data_for_db)
            db.add(profile)
            db.commit()
            db.refresh(profile)
            return profile.id
    finally:
        db.close()

def get_user_profile(mobile_number):
    """Get user profile from database"""
    db = SessionLocal()
    try:
        profile = db.query(MSMEProfile).filter(
            MSMEProfile.mobile_number == mobile_number
        ).first()

        if profile:
            return {
                'id': profile.id,
                'mobile_number': profile.mobile_number,
                'full_name': profile.full_name,
                'company_name': profile.company_name,
                'business_type': profile.business_type,
                'state': profile.state,
                'city': profile.city,
                'verification_status': profile.verification_status,
                'organisation_type': profile.organisation_type,
                'major_activity': profile.major_activity,
                'enterprise_type': profile.enterprise_type
            }
        return None
    finally:
        db.close()

# ==================== ML & ANALYTICS FUNCTIONS ====================

def normalize(series):
    """Normalize series to 0-1 range"""
    if series.empty or series.max() == series.min():
        return pd.Series(0, index=series.index)
    return (series - series.min()) / (series.max() - series.min() + 1e-9)

def calculate_scores(df):
    """Calculate risk and performance scores, including new Performance_Score"""
    column_mapping = {
        'Sales_INR': 'Monthly_Sales_INR',
        'Monthly_Sales': 'Monthly_Sales_INR',
        'Operating_Cost_INR': 'Monthly_Operating_Cost_INR',
        'Operating_Cost': 'Monthly_Operating_Cost_INR',
        'Outstanding_Loan': 'Outstanding_Loan_INR',
        'Vendor_Reliability': 'Vendor_Delivery_Reliability',
        'Inventory_Turnover_Rate': 'Inventory_Turnover',
        'Average_Margin_Percent': 'Avg_Margin_Percent',
        'Monthly_Demand': 'Monthly_Demand_Units',
        'Returns': 'Returns_Percentage'
    }

    for old_name, new_name in column_mapping.items():
        if old_name in df.columns and old_name != new_name:
            df.rename(columns={old_name: new_name}, inplace=True)

    numeric_cols = ['Monthly_Sales_INR', 'Monthly_Operating_Cost_INR', 'Outstanding_Loan_INR',
                   'Vendor_Delivery_Reliability', 'Inventory_Turnover', 'Avg_Margin_Percent',
                   'Monthly_Demand_Units', 'Returns_Percentage']

    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)
        else:
            df[col] = 0

    df['Monthly_Sales_INR_Adjusted'] = df['Monthly_Sales_INR'].replace(0, 1e-9)

    df["Cashflow_Stress"] = normalize(df["Monthly_Operating_Cost_INR"] / df["Monthly_Sales_INR_Adjusted"])
    df["Loan_Stress"] = normalize(df["Outstanding_Loan_INR"] / (df["Monthly_Sales_INR_Adjusted"] * 12))
    df["Financial_Risk_Score"] = (0.5 * df["Cashflow_Stress"] + 0.5 * df["Loan_Stress"]).clip(0, 1)

    df["Vendor_Score"] = (
        0.5 * df["Vendor_Delivery_Reliability"] +
        0.3 * normalize(df["Inventory_Turnover"]) +
        0.2 * normalize(df["Avg_Margin_Percent"])
    ).clip(0, 1)

    df["Growth_Potential_Score"] = (
        0.4 * normalize(df["Monthly_Demand_Units"]) +
        0.35 * normalize(df["Avg_Margin_Percent"]) +
        0.25 * (1 - normalize(df["Returns_Percentage"]))
    ).clip(0, 1)

    df["MSME_Health_Score"] = (
        (1 - df["Financial_Risk_Score"]) * 0.4 +
        df["Vendor_Score"] * 0.3 +
        df["Growth_Potential_Score"] * 0.3
    ) * 100

    df['Profitability_Ratio'] = normalize(df['Avg_Margin_Percent'] * df['Monthly_Sales_INR_Adjusted'])
    df['Operational_Efficiency'] = (1 - normalize(df['Monthly_Operating_Cost_INR'] / df['Monthly_Sales_INR_Adjusted'])).clip(0,1)
    df['Customer_Satisfaction'] = (1 - normalize(df['Returns_Percentage'])).clip(0,1)

    df['Performance_Score'] = (
        0.3 * df['Profitability_Ratio'] +
        0.25 * df['Operational_Efficiency'] +
        0.2 * df['Customer_Satisfaction'] +
        0.15 * df['Vendor_Delivery_Reliability'] +
        0.1 * normalize(df['Inventory_Turnover'])
    ).clip(0,1) * 100

    return df

def forecast_sales(df):
    """Per-store sales forecasting using Prophet or fallback"""
    all_forecasts = {'6_month': {}, '12_month': {}}

    if 'Date' not in df.columns:
        return {
            '6_month': {'forecast': 0, 'lower': 0, 'upper': 0},
            '12_month': {'forecast': 0, 'lower': 0, 'upper': 0}
        }
    df['Date'] = pd.to_datetime(df['Date'])

    if PROPHET_AVAILABLE:
        unique_stores = df['Store_ID'].unique()
        for store_id in unique_stores:
            store_df = df[df['Store_ID'] == store_id].copy()
            ts_data = store_df.groupby('Date')['Monthly_Sales_INR'].sum().reset_index()
            ts_data.columns = ['ds', 'y']

            if len(ts_data) < 2:
                store_sales_sum = store_df['Monthly_Sales_INR'].sum()
                avg_growth = 0.05
                all_forecasts['6_month'][store_id] = {
                    'forecast': store_sales_sum * 6 * (1 + avg_growth),
                    'lower': store_sales_sum * 6 * (1 + avg_growth) * 0.85,
                    'upper': store_sales_sum * 6 * (1 + avg_growth) * 1.15
                }
                all_forecasts['12_month'][store_id] = {
                    'forecast': store_sales_sum * 12 * (1 + avg_growth),
                    'lower': store_sales_sum * 12 * (1 + avg_growth) * 0.85,
                    'upper': store_sales_sum * 12 * (1 + avg_growth) * 1.15
                }
                continue

            try:
                model = Prophet(
                    yearly_seasonality=True,
                    weekly_seasonality=False,
                    daily_seasonality=False,
                    changepoint_prior_scale=0.05
                )
                model.fit(ts_data)

                future_6m = model.make_future_dataframe(periods=180)
                forecast_6m_df = model.predict(future_6m)

                future_12m = model.make_future_dataframe(periods=365)
                forecast_12m_df = model.predict(future_12m)

                all_forecasts['6_month'][store_id] = {
                    'forecast': forecast_6m_df['yhat'].tail(180).sum(),
                    'lower': forecast_6m_df['yhat_lower'].tail(180).sum(),
                    'upper': forecast_6m_df['yhat_upper'].tail(180).sum()
                }
                all_forecasts['12_month'][store_id] = {
                    'forecast': forecast_12m_df['yhat'].tail(365).sum(),
                    'lower': forecast_12m_df['yhat_lower'].tail(365).sum(),
                    'upper': forecast_12m_df['yhat_upper'].tail(365).sum()
                }
            except Exception as e:
                store_sales_sum = store_df['Monthly_Sales_INR'].sum()
                avg_growth = 0.05
                all_forecasts['6_month'][store_id] = {
                    'forecast': store_sales_sum * 6 * (1 + avg_growth),
                    'lower': store_sales_sum * 6 * (1 + avg_growth) * 0.85,
                    'upper': store_sales_sum * 6 * (1 + avg_growth) * 1.15
                }
                all_forecasts['12_month'][store_id] = {
                    'forecast': store_sales_sum * 12 * (1 + avg_growth),
                    'lower': store_sales_sum * 12 * (1 + avg_growth) * 0.85,
                    'upper': store_sales_sum * 12 * (1 + avg_growth) * 1.15
                }
    else:
        total_sales = df['Monthly_Sales_INR'].sum()
        avg_growth = 0.05
        fallback_6m = total_sales * 6 * (1 + avg_growth)
        fallback_12m = total_sales * 12 * (1 + avg_growth)
        return {
            '6_month': {'forecast': fallback_6m, 'lower': fallback_6m * 0.85, 'upper': fallback_6m * 1.15},
            '12_month': {'forecast': fallback_12m, 'lower': fallback_12m * 0.85, 'upper': fallback_12m * 1.15}
        }

    if all_forecasts['6_month']:
        total_forecast_6m = sum(f['forecast'] for f in all_forecasts['6_month'].values())
        total_lower_6m = sum(f['lower'] for f in all_forecasts['6_month'].values())
        total_upper_6m = sum(f['upper'] for f in all_forecasts['6_month'].values())
    else:
        total_forecast_6m, total_lower_6m, total_upper_6m = 0, 0, 0

    if all_forecasts['12_month']:
        total_forecast_12m = sum(f['forecast'] for f in all_forecasts['12_month'].values())
        total_lower_12m = sum(f['lower'] for f in all_forecasts['12_month'].values())
        total_upper_12m = sum(f['upper'] for f in all_forecasts['12_month'].values())
    else:
        total_forecast_12m, total_lower_12m, total_upper_12m = 0, 0, 0

    return {
        '6_month': {'forecast': total_forecast_6m, 'lower': total_lower_6m, 'upper': total_upper_6m},
        '12_month': {'forecast': total_forecast_12m, 'lower': total_lower_12m, 'upper': total_upper_12m},
        'per_store_forecasts': all_forecasts
    }

def segment_customers(df):
    """K-Means customer segmentation"""
    try:
        if 'SKU_Name' not in df.columns:
            return None

        if 'Date' in df.columns:
            df['Date'] = pd.to_datetime(df['Date'])
            reference_date = df['Date'].max()

            rfm = df.groupby('SKU_Name').agg({
                'Date': lambda x: (reference_date - x.max()).days,
                'Monthly_Sales_INR': ['count', 'sum']
            })

            rfm.columns = ['recency', 'frequency', 'monetary']

            if len(rfm) >= 3:
                scaler = StandardScaler()
                rfm_scaled = scaler.fit_transform(rfm)

                n_clusters = min(5, len(rfm))
                if n_clusters == 0:
                    return None
                kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init='auto')
                rfm['segment'] = kmeans.fit_predict(rfm_scaled)

                segment_names = ['Champions', 'Loyal', 'Potential', 'At Risk', 'Lost']
                rfm['segment_name'] = rfm['segment'].apply(
                    lambda x: segment_names[x] if x < len(segment_names) else f'Segment {x}'
                )

                return rfm['segment_name'].value_counts().to_dict()

        return None
    except Exception as e:
        return None

def generate_insights(user_data, df):
    """Generate AI-powered insights"""
    try:
        company_name = user_data.get('company_name', 'Your Company')
        df = calculate_scores(df)

        total_sales = df['Monthly_Sales_INR'].sum()
        total_products = len(df['SKU_Name'].unique()) if 'SKU_Name' in df.columns else 0
        avg_margin = df['Avg_Margin_Percent'].mean()
        overall_performance_score = df['Performance_Score'].mean()

        total_sales_display = f"₹{total_sales:,.0f}" if not pd.isna(total_sales) else "N/A"
        avg_margin_display = f"{avg_margin:.1f}" if not pd.isna(avg_margin) else "N/A"
        overall_performance_score_display = f"{overall_performance_score:.1f}" if not pd.isna(overall_performance_score) else "N/A"
        msme_health_score_display = f"{df['MSME_Health_Score'].mean():.1f}" if not pd.isna(df["MSME_Health_Score"].mean()) else "N/A"

        if 'SKU_Name' in df.columns and not df.empty and not df['Monthly_Sales_INR'].empty:
            top_skus = df.nlargest(5, 'Monthly_Sales_INR')[['SKU_Name', 'Monthly_Sales_INR', 'Monthly_Demand_Units']]
        else:
            top_skus = pd.DataFrame(columns=['SKU_Name', 'Monthly_Sales_INR', 'Monthly_Demand_Units'])

        insights = f"""# 🎯 AI-Powered Business Insights for {company_name}

## 📊 Overall Performance Summary
- **Total Sales:** {total_sales_display}
- **Total Products Analyzed:** {total_products}
- **Average Profit Margin:** {avg_margin_display}%
- **Overall MSME Health Score:** {msme_health_score_display}%
- **Overall Performance Score:** {overall_performance_score_display}%

## 🏆 Top 5 Performing Products
"""
        if not top_skus.empty:
            for idx, row in top_skus.iterrows():
                sku_sales = f"₹{row['Monthly_Sales_INR']:,.0f}" if not pd.isna(row['Monthly_Sales_INR']) else "N/A"
                sku_demand = f"{row['Monthly_Demand_Units']:.0f}" if not pd.isna(row['Monthly_Demand_Units']) else "N/A"
                insights += f"\n{idx+1}. **{row['SKU_Name']}** - {sku_sales} | {sku_demand} units"
        else:
            insights += "\nNo SKU data available for top products."

        financial_risk_score_display = f"{df['Financial_Risk_Score'].mean():.2f}" if not pd.isna(df["Financial_Risk_Score"].mean()) else "N/A"
        vendor_score_display = f"{df['Vendor_Score'].mean():.2f}" if not pd.isna(df["Vendor_Score"].mean()) else "N/A"
        growth_potential_score_display = f"{df['Growth_Potential_Score'].mean():.2f}" if not pd.isna(df["Growth_Potential_Score"].mean()) else "N/A"

        insights += f"""

## 📈 Performance Metrics
- **Financial Risk Score:** {financial_risk_score_display} (Lower is better)
- **Vendor Reliability Score:** {vendor_score_display}
- **Growth Potential Score:** {growth_potential_score_display}
"""

        forecast_results = forecast_sales(df)
        forecast_6m_forecast = f"₹{forecast_results['6_month']['forecast']:,.0f}" if not pd.isna(forecast_results['6_month']['forecast']) else "N/A"
        forecast_6m_lower = f"₹{forecast_results['6_month']['lower']:,.0f}" if not pd.isna(forecast_results['6_month']['lower']) else "N/A"
        forecast_6m_upper = f"₹{forecast_results['6_month']['upper']:,.0f}" if not pd.isna(forecast_results['6_month']['upper']) else "N/A"
        forecast_12m_forecast = f"₹{forecast_results['12_month']['forecast']:,.0f}" if not pd.isna(forecast_results['12_month']['forecast']) else "N/A"
        forecast_12m_lower = f"₹{forecast_results['12_month']['lower']:,.0f}" if not pd.isna(forecast_results['12_month']['lower']) else "N/A"
        forecast_12m_upper = f"₹{forecast_results['12_month']['upper']:,.0f}" if not pd.isna(forecast_results['12_month']['upper']) else "N/A"

        insights += f"""

## 🔮 ML-Powered Sales Forecast (Overall Company)
### 6-Month Projection
- **Forecasted Sales:** {forecast_6m_forecast}
- **Expected Range:** {forecast_6m_lower} - {forecast_6m_upper}

### 12-Month Projection
- **Forecasted Sales:** {forecast_12m_forecast}
- **Expected Range:** {forecast_12m_lower} - {forecast_12m_upper}

## 💡 AI-Generated Recommendations

### 🎯 Immediate Actions
1. **Prioritize Top Performers:** Focus inventory and marketing on your top 5 SKUs (if available).
2. **Risk Mitigation:** Review products with Financial Risk Score > 0.7.
3. **Vendor Management:** Strengthen partnerships with high-reliability vendors.
4. **Improve Customer Satisfaction:** Analyze products with high Returns_Percentage to reduce returns.

### 📊 Strategic Initiatives
5. **Demand Forecasting:** Use ML predictions to optimize stock levels and reduce waste.
6. **Margin Optimization:** Analyze low-margin products for pricing or cost reduction opportunities.
7. **Growth Planning:** Invest in products showing high growth potential scores.
8. **Enhance Operational Efficiency:** Identify areas to reduce Monthly Operating Cost relative to sales.
"""

        segments = segment_customers(df)
        if segments:
            insights += "\n\n## 👥 Customer Segments (K-Means ML Clustering)\n"
            for segment, count in segments.items():
                insights += f"- **{segment}:** {count} products\n"

        high_risk = df[df['Financial_Risk_Score'] > 0.7]
        if len(high_risk) > 0:
            insights += f"\n\n## ⚠️ Risk Alerts\n{len(high_risk)} products require immediate attention due to high financial risk.\n"

        if 'Store_ID' in df.columns and forecast_results.get('per_store_forecasts'):
            insights += "\n\n## 📈 Store-Specific Sales Forecasts\n"
            for store_id, forecasts in forecast_results['per_store_forecasts']['6_month'].items():
                store_forecast_val = f"₹{forecasts['forecast']:,.0f}" if not pd.isna(forecasts['forecast']) else "N/A"
                store_lower_val = f"₹{forecasts['lower']:,.0f}" if not pd.isna(forecasts['lower']) else "N/A"
                store_upper_val = f"₹{forecasts['upper']:,.0f}" if not pd.isna(forecasts['upper']) else "N/A"
                insights += f"- **Store {store_id} (6-month):** Forecast: {store_forecast_val} (Range: {store_lower_val} - {store_upper_val})\n"

        return insights, None, forecast_results

    except Exception as e:
        import traceback
        error_msg = f"Error generating insights: {str(e)}\n\n{traceback.format_exc()}\n\nPlease ensure your Excel file has the required columns."
        return None, error_msg, None

def generate_dashboard_data(user_data, df):
    """Generate dashboard KPIs and charts"""
    try:
        df = calculate_scores(df)

        total_sales = df['Monthly_Sales_INR'].sum()
        avg_margin = df['Avg_Margin_Percent'].mean() if 'Avg_Margin_Percent' in df.columns else np.nan
        total_profit = total_sales * (avg_margin / 100) if not pd.isna(avg_margin) else np.nan

        health_score = df['MSME_Health_Score'].mean()
        growth_score = df['Growth_Potential_Score'].mean()
        performance_score = df['Performance_Score'].mean()

        total_sales_display = f"₹{total_sales:,.0f}" if not pd.isna(total_sales) else "N/A"
        total_profit_display = f"₹{total_profit:,.0f}" if not pd.isna(total_profit) else "N/A"
        health_score_display = f"{health_score:.1f}%" if not pd.isna(health_score) else "N/A"
        growth_score_display = f"{growth_score:.2f}" if not pd.isna(growth_score) else "N/A"
        performance_score_display = f"{performance_score:.1f}%" if not pd.isna(performance_score) else "N/A"

        plt.style.use('seaborn-v0_8-darkgrid')
        fig1, ax1 = plt.subplots(figsize=(10, 6))
        if 'SKU_Name' in df.columns and not df.empty:
            top_skus = df.nlargest(10, 'Monthly_Sales_INR')
            colors = plt.cm.viridis(np.linspace(0.3, 0.9, len(top_skus)))
            bars = ax1.barh(top_skus['SKU_Name'], top_skus['Monthly_Sales_INR'], color=colors)
            ax1.set_xlabel('Sales (INR)', fontsize=12, fontweight='bold')
            ax1.set_title('Top 10 Products by Sales Revenue', fontsize=14, fontweight='bold', pad=20)
            ax1.grid(axis='x', alpha=0.3)
            for i, bar in enumerate(bars):
                width = bar.get_width()
                ax1.text(width, bar.get_y() + bar.get_height()/2, f'₹{width:,.0f}',
                        ha='left', va='center', fontsize=9, fontweight='bold')
        else:
            ax1.text(0.5, 0.5, 'No SKU data to display', horizontalalignment='center', verticalalignment='center', transform=ax1.transAxes, fontsize=14)
            ax1.set_title('Top 10 Products by Sales Revenue (No Data)', fontsize=14, fontweight='bold', pad=20)
        plt.tight_layout()

        fig2, ax2 = plt.subplots(figsize=(10, 6))
        scores_labels = ['Financial\nRisk', 'Vendor\nScore', 'Growth\nPotential', 'Performance\nScore']
        values = [
            df['Financial_Risk_Score'].mean(),
            df['Vendor_Score'].mean(),
            df['Growth_Potential_Score'].mean(),
            df['Performance_Score'].mean() / 100
        ]
        values = [0 if pd.isna(v) else v for v in values]

        colors = ['#e74c3c', '#2ecc71', '#3498db', '#f39c12']
        bars = ax2.bar(scores_labels, values, color=colors, alpha=0.8, width=0.6)
        ax2.set_ylabel('Score (0-1)', fontsize=12, fontweight='bold')
        ax2.set_title('Performance Scores Overview', fontsize=14, fontweight='bold', pad=20)
        ax2.set_ylim(0, 1)
        ax2.grid(axis='y', alpha=0.3)
        for bar, val in zip(bars, values):
            height = bar.get_height()
            ax2.text(bar.get_x() + bar.get_width()/2., height,
                    f'{val:.2f}', ha='center', va='bottom', fontsize=11, fontweight='bold')
        plt.tight_layout()

        fig3, ax3 = plt.subplots(figsize=(10, 6))
        if not df.empty and 'Monthly_Sales_INR' in df.columns and 'Avg_Margin_Percent' in df.columns:
            scatter = ax3.scatter(df['Monthly_Sales_INR'], df['Avg_Margin_Percent'],
                       alpha=0.6, c=df['MSME_Health_Score'], cmap='RdYlGn',
                       s=100, edgecolors='black', linewidth=0.5)
            ax3.set_xlabel('Monthly Sales (INR)', fontsize=12, fontweight='bold')
            ax3.set_ylabel('Profit Margin (%)', fontsize=12, fontweight='bold')
            ax3.set_title('Sales vs Margin Analysis (Color = Health Score)', fontsize=14, fontweight='bold', pad=20)
            ax3.grid(True, alpha=0.3)
            cbar = plt.colorbar(scatter, ax=ax3)
            cbar.set_label('Health Score', fontsize=11, fontweight='bold')
        else:
            ax3.text(0.5, 0.5, 'No sales/margin data to display', horizontalalignment='center', verticalalignment='center', transform=ax3.transAxes, fontsize=14)
            ax3.set_title('Sales vs Margin Analysis (No Data)', fontsize=14, fontweight='bold', pad=20)
        plt.tight_layout()

        fig4, ax4 = plt.subplots(figsize=(10, 6))
        forecast_results = forecast_sales(df)
        months = ['6-Month\nForecast', '12-Month\nForecast']
        forecasts = [
            forecast_results['6_month']['forecast'],
            forecast_results['12_month']['forecast']
        ]
        lowers = [
            forecast_results['6_month']['lower'],
            forecast_results['12_month']['lower']
        ]
        uppers = [
            forecast_results['12_month']['upper'],
            forecast_results['12_month']['upper']
        ]

        x_pos = np.arange(len(months))
        bars = ax4.bar(x_pos, forecasts, color=['#9b59b6', '#e67e22'], alpha=0.8, width=0.5)

        errors = [[forecasts[i] - lowers[i] for i in range(len(forecasts))],
                 [uppers[i] - forecasts[i] for i in range(len(forecasts))]]
        ax4.errorbar(x_pos, forecasts, fmt='none', ecolor='black',
                    capsize=5, capthick=2, alpha=0.5)

        ax4.set_ylabel('Forecasted Sales (INR)', fontsize=12, fontweight='bold')
        ax4.set_title('ML Sales Forecast with Confidence Intervals (Overall)', fontsize=14, fontweight='bold', pad=20)
        ax4.set_xticks(x_pos)
        ax4.set_xticklabels(months)
        ax4.grid(axis='y', alpha=0.3)

        for i, (bar, val) in enumerate(zip(bars, forecasts)):
            height = bar.get_height()
            ax4.text(bar.get_x() + bar.get_width()/2., height,
                    f'₹{val:,.0f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

        plt.tight_layout()

        return (
            f"### 💰 Total Sales\n# {total_sales_display}",
            f"### 📈 Total Profit\n# {total_profit_display}",
            f"### 🧠 Health Score\n# {health_score_display}",
            f"### 🚀 Growth Score\n# {growth_score_display}",
            f"### 🌟 Performance Score\n# {performance_score_display}",
            fig1, fig2, fig3, fig4,
            None
        )

    except Exception as e:
        import traceback
        traceback.print_exc()
        return ("N/A", "N/A", "N/A", "N/A", "N/A", None, None, None, None, f"Error: {str(e)}")

def generate_pdf_report(user_data, df, dashboard_figs=None):
    """Generate comprehensive storyboard-style PDF report with humanized insights and recommendations"""
    try:
        from matplotlib.backends.backend_pdf import PdfPages
        import textwrap

        df = calculate_scores(df)
        forecast_results = forecast_sales(df)

        pdf_path = f"/tmp/{user_data.get('company_name', 'Company')}_report.pdf".replace(" ", "_")

        # Calculate key metrics
        total_sales = df['Monthly_Sales_INR'].sum()
        avg_margin = df['Avg_Margin_Percent'].mean()
        total_profit = total_sales * (avg_margin / 100) if not pd.isna(avg_margin) else 0
        health_score = df['MSME_Health_Score'].mean()
        performance_score = df['Performance_Score'].mean()
        risk_score = df['Financial_Risk_Score'].mean()
        growth_score = df['Growth_Potential_Score'].mean()
        vendor_score = df['Vendor_Score'].mean()

        # Get top and bottom performers
        if 'SKU_Name' in df.columns and not df.empty:
            top_3_products = df.nlargest(3, 'Monthly_Sales_INR')[['SKU_Name', 'Monthly_Sales_INR', 'Avg_Margin_Percent']]
            bottom_3_products = df.nsmallest(3, 'Monthly_Sales_INR')[['SKU_Name', 'Monthly_Sales_INR', 'Avg_Margin_Percent']]
        else:
            top_3_products = pd.DataFrame()
            bottom_3_products = pd.DataFrame()

        with PdfPages(pdf_path) as pdf:
            # ============ PAGE 1: COVER PAGE ============
            fig = plt.figure(figsize=(8.5, 11))
            plt.axis('off')

            plt.text(0.5, 0.75, "Business Intelligence Report", fontsize=28, ha='center', weight='bold', color='#0f2557')
            plt.text(0.5, 0.68, f"{user_data.get('company_name', 'Your Company')}", fontsize=20, ha='center', color='#1a3a6b')
            plt.text(0.5, 0.62, f"Prepared on {datetime.datetime.now().strftime('%B %d, %Y')}", fontsize=12, ha='center', color='#555')

            plt.text(0.5, 0.50, "Your Personalized Growth Roadmap", fontsize=16, ha='center', style='italic', color='#2c5282')
            plt.text(0.5, 0.45, "Powered by AI-Driven Analytics", fontsize=11, ha='center', color='#666')

            # Add decorative line
            plt.plot([0.2, 0.8], [0.40, 0.40], 'k-', linewidth=2, alpha=0.3)

            plt.text(0.5, 0.30, "Inside This Report:", fontsize=14, ha='center', weight='bold', color='#0f2557')
            report_sections = [
                "• Executive Summary & Key Insights",
                "• Your Business Health Analysis",
                "• Revenue & Growth Opportunities",
                "• Marketing & Sales Strategies",
                "• Operational Improvements",
                "• Financial Health & Risk Management",
                "• Action Plan for Success"
            ]
            y_pos = 0.26
            for section in report_sections:
                plt.text(0.3, y_pos, section, fontsize=10, ha='left', color='#333')
                y_pos -= 0.03

            plt.text(0.5, 0.05, "DataNetra.ai - Your AI-Powered Business Partner", fontsize=9, ha='center', color='#888', style='italic')

            pdf.savefig(fig, bbox_inches='tight')
            plt.close(fig)

            # ============ PAGE 2: EXECUTIVE SUMMARY ============
            fig = plt.figure(figsize=(8.5, 11))
            plt.axis('off')

            plt.text(0.5, 0.95, "Executive Summary", fontsize=22, ha='center', weight='bold', color='#0f2557')
            plt.plot([0.15, 0.85], [0.93, 0.93], 'k-', linewidth=1.5, alpha=0.3)

            summary_intro = f"""Dear {user_data.get('full_name', 'Business Owner')},

Congratulations on taking the first step toward data-driven growth! After analyzing your business data, we're excited to share insights that will help {user_data.get('company_name', 'your company')} reach new heights.

Here's what we discovered about your business:"""

            wrapped_intro = textwrap.fill(summary_intro, width=80)
            y_pos = 0.88
            for line in wrapped_intro.split('\n'):
                plt.text(0.15, y_pos, line, fontsize=10, ha='left', color='#333', family='serif')
                y_pos -= 0.025

            # Key Metrics Box
            y_pos -= 0.03
            plt.text(0.5, y_pos, "📊 Your Business at a Glance", fontsize=14, ha='center', weight='bold', color='#0f2557')
            y_pos -= 0.04

            metrics = [
                f"Total Revenue: ₹{total_sales:,.0f}",
                f"Estimated Profit: ₹{total_profit:,.0f}",
                f"Business Health Score: {health_score:.1f}/100",
                f"Performance Score: {performance_score:.1f}/100",
                f"Growth Potential: {growth_score:.2f}/1.0"
            ]

            for metric in metrics:
                plt.text(0.20, y_pos, f"✓ {metric}", fontsize=11, ha='left', color='#2d3748', weight='bold')
                y_pos -= 0.035

            # The Story
            y_pos -= 0.04

            if health_score >= 70:
                health_narrative = f"Your business is performing strongly with a health score of {health_score:.1f}/100. You're doing many things right, and we've identified specific opportunities to take you even further."
            elif health_score >= 50:
                health_narrative = f"Your business shows solid potential with a health score of {health_score:.1f}/100. With the right strategic adjustments, you're positioned for significant growth."
            else:
                health_narrative = f"Your business has room for improvement with a health score of {health_score:.1f}/100. Don't worry—we've identified clear actions that can quickly strengthen your position."

            story_text = f"""{health_narrative}

Our AI has analyzed every aspect of your operations—from sales patterns to vendor relationships, from profit margins to customer behavior. What we found is a business with unique strengths and specific opportunities for improvement.

The next pages contain your personalized roadmap: actionable strategies to increase revenue, improve efficiency, and build lasting competitive advantage in your market."""

            wrapped_story = textwrap.fill(story_text, width=80)
            for line in wrapped_story.split('\n'):
                plt.text(0.15, y_pos, line, fontsize=10, ha='left', color='#333', family='serif')
                y_pos -= 0.025

            pdf.savefig(fig, bbox_inches='tight')
            plt.close(fig)

            # ============ PAGE 3: REVENUE INSIGHTS & OPPORTUNITIES ============
            fig = plt.figure(figsize=(8.5, 11))
            plt.axis('off')

            plt.text(0.5, 0.95, "💰 Revenue Insights & Growth Opportunities", fontsize=20, ha='center', weight='bold', color='#0f2557')
            plt.plot([0.15, 0.85], [0.93, 0.93], 'k-', linewidth=1.5, alpha=0.3)

            y_pos = 0.89

            # Revenue forecast
            forecast_6m = forecast_results['6_month']['forecast']
            forecast_12m = forecast_results['12_month']['forecast']

            revenue_section = f"""🎯 REVENUE PROJECTION

Our AI forecasting model predicts exciting growth potential:

• 6-Month Forecast: ₹{forecast_6m:,.0f}
  This represents your expected revenue if you maintain current performance.

• 12-Month Forecast: ₹{forecast_12m:,.0f}
  Your annual trajectory shows {'strong' if forecast_12m > total_sales * 12 else 'steady'} growth potential."""

            for line in revenue_section.split('\n'):
                plt.text(0.15, y_pos, line, fontsize=10, ha='left', color='#2d3748')
                y_pos -= 0.025

            y_pos -= 0.02

            # Top performers
            plt.text(0.15, y_pos, "🏆 YOUR STAR PERFORMERS", fontsize=12, ha='left', weight='bold', color='#0f2557')
            y_pos -= 0.03

            if not top_3_products.empty:
                plt.text(0.15, y_pos, "These products are driving your success:", fontsize=10, ha='left', color='#333')
                y_pos -= 0.025

                for idx, row in top_3_products.iterrows():
                    product_text = f"{idx+1}. {row['SKU_Name']} - ₹{row['Monthly_Sales_INR']:,.0f} ({row['Avg_Margin_Percent']:.1f}% margin)"
                    plt.text(0.20, y_pos, product_text, fontsize=9, ha='left', color='#2d3748')
                    y_pos -= 0.022

                y_pos -= 0.02
                revenue_recommendation = """💡 RECOMMENDATION: Double down on these winners!
• Increase inventory for your top performers
• Create bundle offers featuring these products
• Use them as loss leaders to attract new customers
• Study what makes them successful and apply those lessons to other products"""

                for line in revenue_recommendation.split('\n'):
                    plt.text(0.15, y_pos, line, fontsize=9, ha='left', color='#2d3748')
                    y_pos -= 0.022

            y_pos -= 0.03

            # Underperformers
            plt.text(0.15, y_pos, "⚠️ UNDERPERFORMING PRODUCTS", fontsize=12, ha='left', weight='bold', color='#c53030')
            y_pos -= 0.03

            if not bottom_3_products.empty:
                plt.text(0.15, y_pos, "These products need attention:", fontsize=10, ha='left', color='#333')
                y_pos -= 0.025

                for idx, row in bottom_3_products.iterrows():
                    product_text = f"• {row['SKU_Name']} - ₹{row['Monthly_Sales_INR']:,.0f} ({row['Avg_Margin_Percent']:.1f}% margin)"
                    plt.text(0.20, y_pos, product_text, fontsize=9, ha='left', color='#2d3748')
                    y_pos -= 0.022

                y_pos -= 0.02
                underperformer_advice = """💡 TURNAROUND STRATEGY:
• Analyze why these products underperform (pricing, placement, marketing?)
• Consider promotional campaigns or price adjustments
• Bundle with popular items to increase visibility
• If margins are too low, renegotiate with suppliers or discontinue"""

                for line in underperformer_advice.split('\n'):
                    plt.text(0.15, y_pos, line, fontsize=9, ha='left', color='#2d3748')
                    y_pos -= 0.022

            pdf.savefig(fig, bbox_inches='tight')
            plt.close(fig)

            # ============ PAGE 4: MARKETING & SALES STRATEGIES ============
            fig = plt.figure(figsize=(8.5, 11))
            plt.axis('off')

            plt.text(0.5, 0.95, "📢 Marketing & Sales Strategies", fontsize=20, ha='center', weight='bold', color='#0f2557')
            plt.plot([0.15, 0.85], [0.93, 0.93], 'k-', linewidth=1.5, alpha=0.3)

            y_pos = 0.89

            marketing_intro = f"""Based on your business type ({user_data.get('business_type', 'retail')}), here's your personalized marketing playbook to increase sales and customer loyalty:"""

            plt.text(0.15, y_pos, marketing_intro, fontsize=10, ha='left', color='#333', style='italic')
            y_pos -= 0.04

            # Marketing strategies based on business performance
            strategies = []

            if avg_margin > 15:
                strategies.append("""🎯 STRATEGY 1: PREMIUM POSITIONING\nYour healthy margins suggest quality products. Leverage this!\n\nActions:\n• Emphasize product quality and unique features in marketing\n• Create premium customer experiences (packaging, service)\n• Use customer testimonials and reviews prominently\n• Consider loyalty programs for repeat customers"""
)
            else:
                strategies.append("""🎯 STRATEGY 1: VALUE OPTIMIZATION\nFocus on volume and efficiency to improve margins.\n\nActions:\n• Highlight competitive pricing in your marketing\n• Create bundle deals to increase average transaction value\n• Negotiate better supplier terms to improve margins\n• Consider dynamic pricing during peak seasons"""
)

            if growth_score > 0.6:
                strategies.append("""🚀 STRATEGY 2: EXPANSION & SCALE\nYour growth potential is high—time to accelerate!\n\nActions:\n• Expand your product range in winning categories\n• Explore new customer segments or geographic areas\n• Invest in digital marketing (social media, Google ads)\n• Consider partnerships or wholesale opportunities\n• Build an email list for direct marketing"""
)
            else:
                strategies.append("""📈 STRATEGY 2: CUSTOMER RETENTION\nFocus on keeping and deepening existing customer relationships.\n\nActions:\n• Launch a customer loyalty or rewards program\n• Send personalized offers based on purchase history\n• Improve customer service and response times\n• Create a referral program (customers bring friends)\n• Regular email updates with exclusive deals"""
)

            strategies.append("""💻 STRATEGY 3: DIGITAL PRESENCE\nModern customers shop online—meet them there!\n\nActions:\n• Optimize your Google Business Profile (if you have one)\n• Be active on social media (Instagram, Facebook, WhatsApp Business)\n• Share behind-the-scenes content and product stories\n• Run targeted social media ads to local customers\n• Encourage online reviews and respond to all feedback"""
)

            strategies.append("""🎁 STRATEGY 4: PROMOTIONAL CALENDAR\nCreate urgency and excitement with planned promotions.\n\nActions:\n• Monthly flash sales on slow-moving inventory\n• Seasonal campaigns (festivals, holidays, back-to-school)\n• First-time customer discounts\n• Weekend special offers\n• Birthday month discounts for loyalty members"""
)

            for strategy in strategies:
                for line in strategy.split('\n'):
                    if y_pos < 0.15:  # Page break needed
                        pdf.savefig(fig, bbox_inches='tight')
                        plt.close(fig)

                        fig = plt.figure(figsize=(8.5, 11))
                        plt.axis('off')
                        y_pos = 0.95

                    plt.text(0.15, y_pos, line, fontsize=9, ha='left', color='#2d3748')
                    y_pos -= 0.022
                y_pos -= 0.02

            pdf.savefig(fig, bbox_inches='tight')
            plt.close(fig)

            # ============ PAGE 5: OPERATIONAL EXCELLENCE ============
            fig = plt.figure(figsize=(8.5, 11))
            plt.axis('off')

            plt.text(0.5, 0.95, "⚙️ Operational Improvements", fontsize=20, ha='center', weight='bold', color='#0f2557')
            plt.plot([0.15, 0.85], [0.93, 0.93], 'k-', linewidth=1.5, alpha=0.3)

            y_pos = 0.89

            operational_intro = """Small operational improvements can dramatically increase your profitability. Here's where to focus:"""
            plt.text(0.15, y_pos, operational_intro, fontsize=10, ha='left', color='#333', style='italic')
            y_pos -= 0.04

            operational_sections = []

            if vendor_score < 0.7:
                operational_sections.append(f"""🤝 VENDOR RELATIONSHIP MANAGEMENT\nYour vendor reliability score is {vendor_score:.2f}/1.0 - there's room for improvement.\n\nAction Steps:\n• Diversify suppliers to reduce dependency on any single vendor\n• Negotiate payment terms that improve your cash flow\n• Set clear quality standards and performance metrics\n• Build backup supplier relationships for critical items\n• Consider vendor consolidation for better pricing power\n• Regular vendor performance reviews"""
)
            else:
                operational_sections.append(f"""🤝 VENDOR RELATIONSHIP MANAGEMENT\nYour vendor score of {vendor_score:.2f}/1.0 is strong! Maintain these relationships.\n\nAction Steps:\n• Reward top-performing vendors with more business\n• Explore co-marketing opportunities with key suppliers\n• Negotiate volume discounts as your business grows\n• Request exclusivity or first-access to new products\n• Build long-term partnerships for stability"""
)

            operational_sections.append("""📦 INVENTORY OPTIMIZATION\nSmart inventory management directly impacts your cash flow and profits.\n\nAction Steps:\n• Use the 80/20 rule: 80% of sales come from 20% of products\n• Stock deeper on fast-movers, lighter on slow-movers\n• Implement just-in-time ordering for perishables\n• Set minimum and maximum stock levels for each product\n• Use our sales forecasts to plan seasonal inventory\n• Regular stock audits to prevent shrinkage and waste\n• Clear out dead stock with aggressive promotions"""
)

            operational_sections.append("""💰 COST REDUCTION OPPORTUNITIES\nEvery rupee saved in costs goes straight to your profit.\n\nAction Steps:\n• Review all recurring expenses monthly (rent, utilities, subscriptions)\n• Negotiate better rates with service providers\n• Reduce energy costs (LED lights, efficient equipment)\n• Minimize waste in operations and inventory\n• Cross-train staff for flexibility\n• Automate repetitive tasks where possible\n• Consider bulk purchasing for frequently used items"""
)

            operational_sections.append("""👥 CUSTOMER EXPERIENCE EXCELLENCE\nHappy customers become repeat customers and refer others.\n\nAction Steps:\n• Train staff on excellent customer service\n• Reduce wait times (checkout, queries, delivery)\n• Make returns and exchanges hassle-free\n• Collect and act on customer feedback\n• Create a welcoming store environment\n• Ensure product availability for popular items\n• Be responsive on phone, WhatsApp, and social media"""
)

            for section in operational_sections:
                for line in section.split('\n'):
                    if y_pos < 0.12:
                        pdf.savefig(fig, bbox_inches='tight')
                        plt.close(fig)

                        fig = plt.figure(figsize=(8.5, 11))
                        plt.axis('off')
                        y_pos = 0.95

                    plt.text(0.15, y_pos, line, fontsize=9, ha='left', color='#2d3748')
                    y_pos -= 0.022
                y_pos -= 0.025

            pdf.savefig(fig, bbox_inches='tight')
            plt.close(fig)

            # ============ PAGE 6: FINANCIAL HEALTH ============
            fig = plt.figure(figsize=(8.5, 11))
            plt.axis('off')

            plt.text(0.5, 0.95, "💵 Financial Health & Risk Management", fontsize=20, ha='center', weight='bold', color='#0f2557')
            plt.plot([0.15, 0.85], [0.93, 0.93], 'k-', linewidth=1.5, alpha=0.3)

            y_pos = 0.89

            if risk_score > 0.7:
                risk_status = "HIGH PRIORITY"
                risk_color = "#c53030"
                risk_message = f"""⚠️ FINANCIAL RISK ALERT\n\nYour financial risk score of {risk_score:.2f}/1.0 requires immediate attention. This suggests cash flow challenges or debt burden that could threaten business stability.\n\nURGENT ACTIONS NEEDED:\n• Review and reduce operating costs immediately\n• Accelerate collections from customers\n• Negotiate extended payment terms with suppliers\n• Consider refinancing high-interest debt\n• Build a cash reserve of at least 2-3 months expenses\n• Avoid taking on new debt until risk improves\n• Monitor cash flow weekly, not monthly"""

            elif risk_score > 0.4:
                risk_status = "MODERATE ATTENTION"
                risk_color = "#d69e2e"
                risk_message = f"""⚡ FINANCIAL STABILITY\n\nYour financial risk score of {risk_score:.2f}/1.0 indicates moderate financial health with some areas needing attention.\n\nRECOMMENDED ACTIONS:\n• Maintain strict expense discipline\n• Build an emergency fund (3-6 months expenses)\n• Improve accounts receivable collection\n• Review and optimize debt structure\n• Create monthly cash flow projections\n• Diversify revenue sources to reduce risk"""

            else:
                risk_status = "HEALTHY"
                risk_color = "#38a169"
                risk_message = f"""✅ STRONG FINANCIAL POSITION\n\nYour financial risk score of {risk_score:.2f}/1.0 indicates healthy financial management. Well done!\n\nMAINTAIN & IMPROVE:\n• Continue prudent financial management\n• Build strategic reserves for growth opportunities\n• Consider growth investments with positive ROI\n• Maintain strong vendor and customer relationships\n• Regular financial health monitoring\n• Plan for seasonal fluctuations"""

            for line in risk_message.split('\n'):
                plt.text(0.15, y_pos, line, fontsize=9, ha='left', color='#2d3748')
                y_pos -= 0.022

            y_pos -= 0.03

            financial_tips = f"""💡 PROFIT MAXIMIZATION TIPS\n\n1. INCREASE AVERAGE TRANSACTION VALUE\n   • Upsell and cross-sell at checkout\n   • Create product bundles\n   • Minimum purchase for free delivery\n   • Tiered pricing (buy more, save more)\n\n2. IMPROVE PROFIT MARGINS\n   • Current average margin: {avg_margin:.1f}%\n   • Renegotiate supplier prices\n   • Reduce operating costs\n   • Premium product positioning\n   • Regular price reviews\n\n3. ACCELERATE CASH FLOW\n   • Offer discounts for early payment\n   • Reduce payment terms to customers\n   • Improve inventory turnover\n   • Clear out slow-moving stock\n   • Multiple payment options (UPI, cards, etc.)\n\n4. SMART GROWTH INVESTMENTS\n   • Invest in your top-performing products\n   • Upgrade customer-facing technology\n   • Marketing that directly drives sales\n   • Staff training for better productivity\n   • Only expand when you have cash reserves"""

            for line in financial_tips.split('\n'):
                if y_pos < 0.12:
                    pdf.savefig(fig, bbox_inches='tight')
                    plt.close(fig)

                    fig = plt.figure(figsize=(8.5, 11))
                    plt.axis('off')
                    y_pos = 0.95

                plt.text(0.15, y_pos, line, fontsize=9, ha='left', color='#2d3748')
                y_pos -= 0.022

            pdf.savefig(fig, bbox_inches='tight')
            plt.close(fig)

            # ============ PAGE 7: ACTION PLAN ============
            fig = plt.figure(figsize=(8.5, 11))
            plt.axis('off')

            plt.text(0.5, 0.95, "🎯 Your 90-Day Action Plan", fontsize=20, ha='center', weight='bold', color='#0f2557')
            plt.plot([0.15, 0.85], [0.93, 0.93], 'k-', linewidth=1.5, alpha=0.3)

            y_pos = 0.89

            action_intro = """Let's turn insights into action! Here's your prioritized roadmap for the next 90 days:"""
            plt.text(0.15, y_pos, action_intro, fontsize=10, ha='left', color='#333', style='italic')
            y_pos -= 0.04

            action_plan = f"""📅 DAYS 1-30: QUICK WINS\n\nWeek 1: Foundation\n□ Review this report with your team\n□ Set specific revenue and profit targets\n□ Audit your current inventory levels\n□ Identify your top 10 best-selling products\n\nWeek 2: Customer Focus\n□ Launch a customer feedback program\n□ Create a customer database (names, phones, emails)\n□ Send a \"thank you\" message to top customers\n□ Plan your first promotional campaign\n\nWeek 3: Operations\n□ Negotiate with top 3 suppliers for better terms\n□ Implement daily sales tracking\n□ Review and reduce wasteful expenses\n□ Train staff on upselling and customer service\n\nWeek 4: Marketing Kickoff\n□ Set up or optimize social media profiles\n□ Create content calendar for next month\n□ Launch first promotional campaign\n□ Collect customer reviews and testimonials\n\n📅 DAYS 31-60: BUILDING MOMENTUM\n\n□ Analyze results from first month's actions\n□ Expand successful marketing initiatives\n□ Introduce 2-3 new promotional offers\n□ Improve top-selling product inventory\n□ Phase out or discount bottom performers\n□ Build email/WhatsApp broadcast list\n□ Create customer loyalty program\n□ Review financial performance weekly\n\n📅 DAYS 61-90: SCALING SUCCESS\n\n□ Double down on what's working\n□ Launch major seasonal campaign\n□ Expand product range in winning categories\n□ Strengthen top vendor relationships\n□ Implement inventory optimization system\n□ Set targets for next quarter\n□ Review and adjust pricing strategy\n□ Plan for future growth investments\n\n🎯 SUCCESS METRICS TO TRACK\n\nWeekly:\n• Total sales revenue\n• Number of transactions\n• Average transaction value\n• Cash in hand\n\nMonthly:\n• Revenue vs target\n• Profit margin percentage\n• Customer acquisition count\n• Inventory turnover\n• Top product performance\n\nQuarterly:\n• Overall business health score\n• Customer retention rate\n• Market share growth\n• Return on marketing investment"""

            for line in action_plan.split('\n'):
                if y_pos < 0.10:
                    pdf.savefig(fig, bbox_inches='tight')
                    plt.close(fig)

                    fig = plt.figure(figsize=(8.5, 11))
                    plt.axis('off')
                    y_pos = 0.95

                font_size = 10 if line.startswith('□') or line.startswith('Week') or line.startswith('•') else 9
                weight = 'bold' if line.startswith('📅') or line.startswith('🎯') else 'normal'
                plt.text(0.15, y_pos, line, fontsize=font_size, ha='left', color='#2d3748', weight=weight)
                y_pos -= 0.025 if font_size == 10 else 0.020

            pdf.savefig(fig, bbox_inches='tight')
            plt.close(fig)

            # ============ PAGE 8: CLOSING & SUPPORT ============
            fig = plt.figure(figsize=(8.5, 11))
            plt.axis('off')

            plt.text(0.5, 0.95, "🌟 Your Journey to Success", fontsize=22, ha='center', weight='bold', color='#0f2557')
            plt.plot([0.15, 0.85], [0.93, 0.93], 'k-', linewidth=1.5, alpha=0.3)

            y_pos = 0.88

            closing_message = f"""Dear {user_data.get('full_name', 'Business Owner')},

Thank you for trusting DataNetra.ai to analyze your business. This report is more than just numbers and charts—it's your personalized roadmap to greater success.

Remember: Every successful business started where you are now. What separates those who thrive from those who survive is taking consistent, informed action.

You have the data. You have the insights. You have the action plan. Now it's time to execute.

KEY TAKEAWAYS:\n\n✓ Your business has a health score of {health_score:.1f}/100\n✓ You're projected to reach ₹{forecast_12m:,.0f} in the next 12 months\n✓ Your top products are performing well—leverage them!\n✓ You have specific opportunities to reduce costs and increase margins\n✓ Your 90-day action plan provides a clear path forward\n\nTHE NEXT STEPS:\n\n1. Share this report with your key team members\n2. Schedule a team meeting to discuss priorities\n3. Start with the \"Quick Wins\" from your action plan\n4. Track your progress weekly\n5. Revisit this report monthly to stay on course\n\nREMEMBER:\n\n\"The best time to plant a tree was 20 years ago.\nThe second best time is today.\"\n\nEvery action you take today builds your business's future. Start small, stay consistent, and watch your business transform.\n\nWe're rooting for your success!\n\n---\n\nNeed Help Implementing These Strategies?\n\n📧 Contact: support@datanetra.ai\n🌐 Visit: www.datanetra.ai\n📱 Follow us on LinkedIn for more business tips\n\n---\n\nThis report was generated using advanced AI and machine learning algorithms specifically trained for MSME business analysis. The recommendations are based on your actual business data and industry best practices.\n\nRegular analysis (monthly or quarterly) will help you track progress and adjust strategies for optimal results.\n\nHere's to your continued growth and success!\n\nWith best wishes,\nThe DataNetra.ai Team\n\nYour AI-Powered Business Partner 🚀"""

            for line in closing_message.split('\n'):
                if y_pos < 0.08:
                    pdf.savefig(fig, bbox_inches='tight')
                    plt.close(fig)

                    fig = plt.figure(figsize=(8.5, 11))
                    plt.axis('off')
                    y_pos = 0.95

                font_size = 10 if line.startswith('✓') or line.startswith('1.') else 9
                style = 'italic' if line.startswith('"') else 'normal'
                weight = 'bold' if any(line.startswith(x) for x in ['KEY TAKEAWAYS', 'THE NEXT STEPS', 'REMEMBER', 'Need Help']) else 'normal'

                plt.text(0.15, y_pos, line, fontsize=font_size, ha='left', color='#2d3748', style=style, weight=weight, family='serif')
                y_pos -= 0.025

            pdf.savefig(fig, bbox_inches='tight')
            plt.close(fig)

            # Add dashboard charts if provided
            if dashboard_figs:
                for fig in dashboard_figs:
                    if fig is not None:
                        pdf.savefig(fig, bbox_inches='tight')

        if not os.path.exists(pdf_path) or os.path.getsize(pdf_path) == 0:
            return None, "PDF generation failed"

        return pdf_path, None

    except Exception as e:
        import traceback
        print(f"PDF generation error: {traceback.format_exc()}")
        return None, f"PDF Error: {str(e)}"

# ==================== MOCK DATA ====================

udyam_master_data = pd.DataFrame({
    'udyam_number': ['UDYAM-UP-01-0000001', 'UDYAM-MH-02-0000002', 'UDYAM-KL-03-0000003'],
    'enterprise_name': ['Tech Innovations Pvt Ltd', 'Retail Solutions Corp', 'FMCG Distributors'],
    'organisation_type': ['Private Limited', 'Partnership', 'Proprietorship'],
    'major_activity': ['Manufacturing', 'Services', 'Trading'],
    'enterprise_type': ['Small', 'Micro', 'Medium'],
    'state': ['Uttar Pradesh', 'Maharashtra', 'Kerala'],
    'city': ['Lucknow', 'Mumbai', 'Kochi']
})

def _fetch_msme_data(msme_number):
    """Fetch MSME data from master database"""
    fetched_data = udyam_master_data[udyam_master_data['udyam_number'] == msme_number]

    if not fetched_data.empty:
        row = fetched_data.iloc[0]
        return (
            row['enterprise_name'],
            row['organisation_type'],
            row['major_activity'],
            row['enterprise_type'],
            row['state'],
            row['city'],
            "✅ MSME Data Fetched Successfully"
        )
    else:
        return "", "", "", "", "", "", "❌ MSME Data Not Found. Please check the number."

# ==================== CUSTOM CSS ====================

custom_css = """
<style>
/* Header styling */
.header-container {
    background: linear-gradient(135deg, #0f2557 0%, #1a3a6b 100%);
    padding: 10px 20px; /* Adjusted padding */
    display: flex;
    justify-content: space-between;
    align-items: center;
    box-shadow: 0 4px 6px rgba(0,0,0,0.3);
}

.top-row {
    display: flex;
    justify-content: space-between;
    align-items: center;
    width: 100%;
    margin-bottom: 30px;
    position: relative;
    z-index: 1;
}

.logo-section {
    display: flex;
    align-items: center; /* Align items horizontally */
    gap: 12px; /* Space between logo and text */
}

.logo-img {
    height: 45px; /* Increased height */
    width: auto;
    filter: brightness(1.1); /* Optional: Adjusts image brightness */
}

.logo-text {
    background: linear-gradient(to right, #6a0dad, #007bff);
    -webkit-background-clip: text;
    -webkit-text-fill-color: transparent;
    color: #6a0dad; /* Fallback for browsers that don't support text gradients */
    font-size: 28px; /* Increased font size */
    font-weight: 700;
    letter-spacing: -0.5px;
}

.nav-menu {
    display: flex;
    gap: 5px; /* Adjusted gap for alignment */
    align-items: center;
    margin-left: auto; /* Align to the right */
}

.nav-link {
    color: white; /* Changed to white for better contrast with header background */
    text-decoration: none;
    font-size: 13px;
    font-weight: 500;
    transition: all 0.3s ease;
    cursor: pointer;
    padding: 2px 5px;
    border-radius: 6px;
    background: rgba(255, 255, 255, 0.1);
    border: 1px solid rgba(255, 255, 255, 0.2);
}

.nav-link:hover {
    background: rgba(255, 255, 255, 0.2);
    border-color: rgba(255, 255, 255, 0.4);
    transform: translateY(-2px);
}

/* Hero section */
.hero-section {
    background: linear-gradient(90deg, #0f172a, #1e3a8a); /* Updated background */
    padding: 60px 40px; /* Updated padding */
    border-radius: 10px; /* Added border-radius */
    text-align: center;
    color: white;
    position: relative;
    overflow: hidden;
}

.hero-section::before {
    content: '';
    position: absolute;
    top: 0;
    left: 0;
    right: 0;
    bottom: 0;
    background: url('data:image/svg+xml,<svg width="100%" height="100%" xmlns="http://www.w3.org/2000/svg"><defs><pattern id="grid" width="40" height="40" patternUnits="userSpaceOnUse"><path d="M 40 0 L 0 0 0 40" fill="none" stroke="rgba(255,255,255,0.05)" stroke-width="1"/></pattern></defs><rect width="100%" height="100%" fill="url(%23grid)"/></svg>');
    opacity: 0.3;
}

.hero-title {
    font-size: 42px; /* Updated font-size */
    font-weight: 700; /* Updated font-weight */
    margin-bottom: 15px;
    position: relative;
    z-index: 1;
    color: white;
}

.hero-section h2.hero-sub-tagline { /* New style for the h2 tagline */
    font-size: 24px;
    margin-top: 10px;
    font-weight: 500;
    color: white;
    opacity: 0.9;
    position: relative;
    z-index: 1;
}

.hero-section p.hero-description { /* New style for the descriptive paragraph */
    margin-top: 15px;
    font-size: 18px;
    opacity: 0.9;
    color: white;
    position: relative;
    z-index: 1;
}

.cta-button { /* New style for the "Get Started" button margin */
    margin-right: 15px;
}

.hero-cta {
    margin-top: 30px;
    display: flex;
    justify-content: center;
    gap: 20px;
}

.btn-hero {
    padding: 12px 30px;
    font-size: 18px;
    font-weight: 600;
    border-radius: 8px;
    cursor: pointer;
    transition: all 0.3s ease;
    border: 2px solid transparent;
}

.btn-primary {
    background-color: #007bff;
    color: white;
    border-color: #007bff;
}

.btn-primary:hover {
    background-color: #0056b3;
    border-color: #0056b3;
    transform: translateY(-2px);
}

.btn-secondary {
    background-color: transparent;
    color: #007bff;
    border-color: #007bff;
}

.btn-secondary:hover {
    background-color: rgba(0, 123, 255, 0.1);
    transform: translateY(-2px);
}

/* How DataNetra Works Section Styling */
#how-datanetra-works-section {
    background: linear-gradient(to bottom, #f0f2f5, #ffffff); /* Light gradient background */
    padding: 40px 30px;
    border-top: 1px solid #e0e0e0; /* Subtle top border */
    border-bottom: 1px solid #e0e0e0; /* Subtle bottom border */
}

#how-datanetra-works-section .section-title {
    text-align: center;
    font-size: 28px;
    font-weight: 700;
    color: #0f2557;
    margin-bottom: 20px;
}

#how-datanetra-works-section > hr { /* Styling for the markdown separator */
    border: 0;
    height: 1px;
    background-image: linear-gradient(to right, rgba(0, 0, 0, 0), rgba(0, 0, 0, 0.1), rgba(0, 0, 0, 0));
    margin: 20px auto;
    width: 80%;
}

.steps-description-column {
    padding: 30px;
    background-color: #ffffff; /* White background for clarity */
    border-radius: 10px;
    box-shadow: 0 4px 15px rgba(0,0,0,0.08);
    border: 1px solid #e0e0e0;
    height: 100%; /* Ensure columns stretch to match height */
}

.steps-description-column h3 {
    color: #2c5282; /* Darker blue for step titles */
    font-size: 20px;
    margin-bottom: 10px;
    border-bottom: 2px solid #e0e0e0;
    padding-bottom: 5px;
}

.steps-description-column > hr { /* Separator between steps */
    border: 0;
    height: 1px;
    background-image: linear-gradient(to right, rgba(0, 0, 0, 0.05), rgba(0, 0, 0, 0.2), rgba(0, 0, 0, 0.05));
    margin: 20px 0;
}

.steps-description-column p {
    font-size: 14px;
    line-height: 1.6;
    color: #555;
}

.login-signup-card { /* Renamed from .login-box */
    /* Removed max-width and margin: 0 auto as Gr.Column handles width */
    padding: 30px;
    background: #ffffff;
    border-radius: 10px;
    box-shadow: 0 4px 15px rgba(0,0,0,0.08);
    border: 1px solid #e0e0e0;
    height: 100%; /* Ensure columns stretch to match height */
    display: flex; /* Use flexbox for internal layout */
    flex-direction: column;
    justify-content: center; /* Center content vertically */
    gap: 15px; /* Space between elements */
}

.login-signup-card .login-signup-title {
    text-align: center;
    color: #0f2557;
    font-size: 22px;
    font-weight: 700;
    margin-bottom: 10px;
}

.login-signup-card .login-signup-instruction {
    text-align: left; /* Specific change */
    color: #0f2557;
    font-size: 22px;
    font-weight: 700;
    margin-bottom: 10px;
    margin-top: 5px; /* Add some top margin for separation */
}

.login-signup-card > hr { /* Separator inside login/signup card */
    border: 0;
    height: 1px;
    background-image: linear-gradient(to right, rgba(0, 0, 0, 0.05), rgba(0, 0, 0, 0.2), rgba(0, 0, 0, 0.05));
    margin: 20px 0;
}

/* Ensure Gradio Row children take equal height */
div.gradio-row {
    align-items: stretch; /* Make children stretch to equal height */
    gap: 20px; /* Space between columns */
}


/* Platform capabilities section */
.capabilities-section {
    background: linear-gradient(to bottom, #f8f9fa, #e9ecef);
    padding: 40px 30px;
}

.section-title {
    text-align: center;
    font-size: 28px;
    font-weight: 700;
    color: #0f2557;
    margin-bottom: 35px;
}

.capabilities-tagline {
    text-align: center;
    font-size: 18px;
    font-weight: 600;
    color: #0f2557; /* Dark blue color */
    margin-top: -20px;
    margin-bottom: 35px;
}

/* Old registration-section styling, keeping for potential other uses but overriding for 'how-datanetra-works-section' */
.section {
    background: white;
    padding: 50px 40px;
    min-height: 400px;
}

/* Old login-box styling, renamed to .login-signup-card */
.login-box {
    max-width: 500px;
    margin: 0 auto;
    background: white;
    padding: 40px;
    border-radius: 12px;
    box-shadow: 0 8px 24px rgba(0,0,0,0.15);
}

.capabilities-grid {
    display: grid;
    grid-template-columns: repeat(4, 1fr);
    gap: 20px;
    max-width: 1400px;
    margin: 0 auto;
}

.capability-card {
    background: white;
    padding: 25px 20px;
    border-radius: 12px;
    box-shadow: 0 4px 12px rgba(0,0,0,0.1);
    transition: all 0.3s ease;
    border: 2px solid transparent;
}

.capability-card:hover {
    transform: translateY(-5px);
    box-shadow: 0 8px 24px rgba(0,0,0,0.15);
    border-color: #4da6ff;
}

.capability-icon {
    font-size: 40px;
    margin-bottom: 15px;
}

.capability-title {
    font-size: 18px;
    font-weight: 700;
    color: #0f2557;
    margin-bottom: 10px;
    line-height: 1.3;
}

.capability-description {
    font-size: 13px;
    color: #555;
    line-height: 1.5;
}

/* Quick Links section */
.quick-links-section {
    background: #ffffff;
    padding: 30px;
    text-align: center;
    border-top: 1px solid #e0e0e0;
}

.quick-links-title {
    font-size: 24px;
    font-weight: 700;
    color: #0f2557;
    margin-bottom: 20px;
}

.quick-links-grid {
    display: flex;
    justify-content: center;
    gap: 50px; /* Increased gap for more space */
    flex-wrap: wrap;
}

.quick-link-item {
    color: #2c5282;
    text-decoration: none;
    font-size: 16px;
    font-weight: 600;
    padding: 10px 25px; /* Increased padding to each link for better click area and visual spacing */
    border-radius: 8px;
    transition: all 0.3s ease;
}

.quick-link-item:hover {
    background-color: #f0f2f5;
    color: #0f2557;
    transform: translateY(-2px);
}

/* Footer */
.footer-section {
    background: #0f2557;
    padding: 5px 20px; /* Reduced padding */
    color: white;
    display: flex;
    justify-content: space-between;
    align-items: center;
    font-size: 13px; /* Reduced overall font size */
}

.footer-left {
    display: flex;
    align-items: center;
    gap: 8px; /* Reduced gap */
}

.footer-lock-icon {
    font-size: 16px; /* Reduced icon size */
    color: white;
}

.footer-text {
    font-size: 13px; /* Reduced font size */
    font-weight: 500;
    color: white;
}

.footer-right {
    display: flex;
    align-items: center;
    gap: 10px; /* Reduced gap */
}

.footer-company {
    font-size: 13px; /* Reduced font size */
    font-weight: 500;
    color: white;
}

.linkedin-link {
    display: flex;
    align-items: center;
    gap: 6px; /* Reduced gap */
    text-decoration: none;
    color: white;
    transition: all 0.3s ease;
    padding: 4px 10px; /* Reduced padding */
    border-radius: 4px; /* Reduced border radius */
    background: rgba(255, 255, 255, 0.1);
    font-size: 12px; /* Reduced font size */
}

.linkedin-link:hover {
    background: rgba(255, 255, 255, 0.2);
    border-color: rgba(255, 255, 255, 0.4);
    transform: translateY(-2px);
}

.linkedin-icon {
    color: white;
    font-size: 16px; /* Reduced icon size */
}

/* Form styling */
.login-card {
    max-width: 450px;
    margin: 40px auto;
    background: white;
    padding: 40px;
    border-radius: 12px;
    box-shadow: 0 8px 24px rgba(0,0,0,0.15);
}

.form-title {
    font-size: 28px;
    font-weight: 700;
    color: #0f2557;
    margin-bottom: 30px;
    text-align: center;
}

/* Gradio overrides */
.gradio-container {
    max-width: 100% !important;
    padding: 0 !important;
}
</style>
"""

# ==================== GRADIO UI ====================

business_types = ["Choose Business Type", "FMCG", "Supermarket", "Clothing", "Electronics", "Manufacturing", "Services"]
roles = ["Business Owner", "Co-Founder", "Manager", "Analyst", "Store Manager"]

with gr.Blocks(title="DataNetra.ai - MSME Intelligence", theme=gr.themes.Soft(), css=custom_css) as demo:

    step_state = gr.State(0)
    user_data_state = gr.State({})
    dashboard_data_state = gr.State({
        'kpi1': "### 💰 Total Sales\n—",
        'kpi2': "### 📈 Total Profit\n—",
        'kpi3': "### 🧠 Health Score\n—",
        'kpi4': "### 🚀 Growth Score\n—",
        'kpi5': "### 🌟 Performance Score\n—",
        'chart1': None,
        'chart2': None,
        'chart3': None,
        'chart4': None
    })

    # Landing Page (Step 0)
    with gr.Column(visible=True) as step0_col:
        gr.HTML("""
<div class=\"header-container\">
    <div class=\"logo-section\">
        <img src=\"https://i.postimg.cc/qRNQYbZJ/Data-Netra-Logo.jpg\"
             class=\"logo-img\"
             alt=\"DataNetra.ai Logo\">
        <div class=\"logo-text\">DataNetra.ai</div>
    </div>

    <div class=\"nav-menu\">
        <!-- Buttons removed as per user request -->
    </div>
</div>

<div class=\"hero-section\">
    <h1 class=\"hero-title\">AI-Powered Retail Intelligence</h1>
    <h2 class=\"hero-sub-tagline\">Data with Vision. Decisions with Confidence.</h2>
    <p class=\"hero-description\">Turn retail data into predictive insights, smarter decisions, and measurable growth.</p>

    <div class=\"hero-cta\">
        <button class=\"btn-hero btn-primary cta-button\">Get Started</button>
        <button class=\"btn-hero btn-secondary\">Request Demo</button>
    </div>
</div>
""")

        # Hidden trigger buttons to capture JS clicks
        show_signup_trigger = gr.Button("", elem_id="show-signup-trigger", visible=False)

        # How DataNetra Works Section (replacing previous quick-login-section)
        with gr.Column(elem_classes="section", elem_id="how-datanetra-works-section"):
            gr.Markdown("## How DataNetra Works", elem_classes="section-title")
            gr.Markdown("---") # Visual separator after title
            with gr.Row():
                # LEFT SIDE – Steps Description
                with gr.Column(scale=1, elem_classes="steps-description-column"):
                    gr.Markdown("### 📥 Step 1: Upload Your Data")
                    gr.Markdown("Easily upload Excel/CSV files (sales, product, operational data) for comprehensive analysis.")

                    gr.Markdown("### 🤖 Step 2: AI-Powered Analysis")
                    gr.Markdown("Our AI processes your data, identifying patterns, forecasting trends, and uncovering hidden insights.")

                    gr.Markdown("### 📊 Step 3: Actionable Dashboards & Recommendations")
                    gr.Markdown("Access interactive dashboards, KPI charts, and personalized recommendations to optimize inventory, boost sales, and drive confident decisions.")

                # RIGHT SIDE – Login/Signup Box
                with gr.Column(scale=1, elem_classes="login-signup-card"):
                    gr.Markdown("**Already Registered**", elem_classes="login-signup-title")
                    quick_login_mobile = gr.Textbox(label="Enter Mobile Number as filled in this Platform", placeholder="+91")
                    quick_login_btn = gr.Button("Login", variant="primary", size="lg")
                    landing_login_error_msg = gr.Markdown(value="", visible=False)
                    gr.Markdown("**First Time User**", elem_classes="login-signup-title")
                    gr.Markdown("**Signup to unlock smart AI Insights**", elem_classes="login-signup-instruction")
                    quick_signup_btn = gr.Button("Sign Up Now", variant="primary", size="lg")


        # Platform Capabilities Section
        gr.HTML("""
        <div class=\"capabilities-section\">
            <h2 class=\"section-title\">Platform Capabilities</h2>
            <p class=\"capabilities-tagline\">Everything you need to make smarter business decisions</p>
            <div class=\"capabilities-grid\">
                <div class=\"capability-card\">
                    <div class=\"capability-icon\">🎯</div>
                    <h3 class=\"capability-title\">Smart Scoring Engine</h3>
                    <p class=\"capability-description\">Automated analysis for accurate performance scoring and health scores</p>
                </div>
                <div class=\"capability-card\">
                    <div class=\"capability-icon\">📊</div>
                    <h3 class=\"capability-title\">Business Health Dashboard</h3>
                    <p class=\"capability-description\">Monitor key metrics and KPIs in one real-time dashboard</p>
                </div>
                <div class=\"capability-card\">
                    <div class=\"capability-icon\">🔮</div>
                    <h3 class=\"capability-title\">Predictive Insights</h3>
                    <p class=\"capability-description\">AI-driven forecasts to anticipate future trends</p>
                </div>
                <div class=\"capability-card\">
                    <div class=\"capability-icon\">🔗</div>
                    <h3 class=\"capability-title\">Easy Integration</h3>
                    <p class=\"capability-description\">Seamlessly connect with your existing retail and POS systems</p>
                </div>
            </div>
        </div>


        <div class=\"quick-links-section\">
            <h3 class=\"quick-links-title\">Quick Links</h3>
            <div class=\"quick-links-grid\">
                <a href=\"#\" class=\"quick-link-item\">About Us</a>
                <a href=\"#\" class=\"quick-link-item\">Features</a>
                <a href=\"#\" class=\"quick-link-item\">Documentation</a>
                <a href=\"#\" class=\"quick-link-item\">Help Center</a>
            </div>
        </div>


        <div class=\"footer-section\">
            <div class=\"footer-left\">
                <span class=\"footer-lock-icon\">🔒</span>
                <span class=\"footer-text\">Data Secured & Protected</span>
            </div>
            <div class=\"footer-right\">
                <span class=\"footer-company\">© 2026 DataNetra.ai</span>
                <a href=\"https://www.linkedin.com/company/108412762/admin/dashboard/\" target=\"_blank\" class=\"linkedin-link\">
                    <span class=\"linkedin-icon\">🔗</span>
                    <span style=\"color: white; font-size: 12px;\">LinkedIn</span>
                </a>
            </div>
        </div>
        """)

    # Login Page (Step 0.5) has been removed from UI

    # Step 1: User Information (Signup)
    with gr.Column(visible=False) as step1_col:
        gr.Markdown("# 📝 Register New User")
        gr.Markdown("## Step 1: User Information")
        name_input = gr.Textbox(label="Full Name*")
        mobile_input = gr.Textbox(label="Mobile Number*")
        email_input = gr.Textbox(label="Email")
        role_input = gr.Dropdown(choices=roles, label="Role*")

        with gr.Row():
            cancel1_btn = gr.Button("Cancel")
            next1_btn = gr.Button("Next →", variant="primary")
        error1 = gr.Markdown()

    # Step 2: MSME Verification
    with gr.Column(visible=False) as step2_col:
        gr.Markdown("## Step 2: MSME Verification")
        msme_number_input = gr.Textbox(label="MSME/Udyam Number*", placeholder="e.g., UDYAM-UP-01-0000001")
        otp_input = gr.Textbox(label="OTP (Enter '1234' for demo)*", type="password")
        fetch_btn = gr.Button("Fetch MSME Data", variant="secondary")
        fetch_status = gr.Markdown()

        gr.Markdown("### Fetched MSME Details")
        fetched_name = gr.Textbox(label="Enterprise Name", interactive=False)
        fetched_org = gr.Textbox(label="Organisation Type", interactive=False)
        fetched_activity = gr.Textbox(label="Major Activity", interactive=False)
        fetched_type = gr.Textbox(label="Enterprise Type", interactive=False)
        fetched_state = gr.Textbox(label="State", interactive=False)
        fetched_city = gr.Textbox(label="City", interactive=False)

        with gr.Row():
            back2_btn = gr.Button("← Back")
            next2_btn = gr.Button("Verify & Next →", variant="primary")
        error2 = gr.Markdown()

    # Step 3: Certificate Validation
    with gr.Column(visible=False) as step3_col:
        gr.Markdown("## Step 3: MSME Certificate Review")

        gr.Markdown("### Confirm MSME Details")
        confirm_name = gr.Textbox(label="Enterprise Name", interactive=False)
        confirm_org = gr.Textbox(label="Organisation Type", interactive=False)
        confirm_activity = gr.Textbox(label="Major Activity", interactive=False)
        confirm_type = gr.Textbox(label="Enterprise Type", interactive=False)
        confirm_state = gr.Textbox(label="State", interactive=False)
        confirm_city = gr.Textbox(label="City", interactive=False)

        consent1 = gr.Checkbox(label="I confirm the above MSME details are correct", value=False)
        consent2 = gr.Checkbox(label="I consent to verify the MSME certificate", value=False)
        certificate_upload = gr.File(label="Upload MSME Certificate (PDF)", file_types=[".pdf"])

        with gr.Row():
            back3_btn = gr.Button("← Back")
            next3_btn = gr.Button("Confirm & Proceed →", variant="primary")
        error3 = gr.Markdown()

    # Step 4: Business Profile
    with gr.Column(visible=False) as step4_col:
        verification_status_display = gr.Markdown(visible=False)
        gr.Markdown("## Step 4: Business Profile")
        business_type_input = gr.Dropdown(choices=business_types, label="Business Type*")
        years_input = gr.Number(label="Years in Operation*", value=1, minimum=0)
        revenue_input = gr.Dropdown(
            label="Monthly Revenue Range*",
            choices=["< 5 Lakh", "5-10 Lakh", "10-50 Lakh", "50 Lakh - 1 Crore", "> 1 Crore"]
        )

        with gr.Row():
            back4_btn = gr.Button("← Back")
            next4_btn = gr.Button("Submit Profile", variant="primary")
        error4 = gr.Markdown()
        proceed_to_step5_btn = gr.Button("Next: Upload Business Data →", variant="primary", visible=False)

    # Step 5: Data Upload & Analysis
    with gr.Column(visible=False) as step5_col:
        login_welcome_message = gr.Markdown(value="", visible=False)
        gr.Markdown("## Step 5: Upload Business Data")
        consent_check = gr.Checkbox(label="I consent to data analysis*", value=False)
        file_upload = gr.File(label="Upload Excel File (.xlsx, .csv)*", file_types=[".xlsx", ".csv"])
        upload_message = gr.Markdown(value="", visible=False)

        with gr.Row():
            back5_btn = gr.Button("← Back")
            cancel5_btn = gr.Button("❌ Cancel", variant="secondary")
            analyze_btn = gr.Button("🚀 Analyze Data", variant="primary")
        error5 = gr.Markdown()

        insights_output = gr.Markdown()
        pdf_output = gr.File(label="📥 Download Your Comprehensive Business Intelligence Report", visible=True)
        view_dashboard_btn = gr.Button("📊 View Dashboard", visible=False, variant="primary")

        # Hidden KPI components for analysis results (to pass to dashboard)
        kpi1 = gr.Markdown(visible=False)
        kpi2 = gr.Markdown(visible=False)
        kpi3 = gr.Markdown(visible=False)
        kpi4 = gr.Markdown(visible=False)
        kpi5 = gr.Markdown(visible=False)
        chart1 = gr.Plot(visible=False)
        chart2 = gr.Plot(visible=False)
        chart3 = gr.Plot(visible=False)
        chart4 = gr.Plot(visible=False)

    # Step 6: Dashboard
    with gr.Column(visible=False) as step6_col:
        gr.Markdown("# 📊 Business Intelligence Dashboard")
        gr.Markdown("## Real-time Analytics & Performance Metrics")

        with gr.Row():
            kpi1_dash = gr.Markdown("### 💰 Total Sales\n—")
            kpi2_dash = gr.Markdown("### 📈 Total Profit\n—")
            kpi3_dash = gr.Markdown("### 🧠 Health Score\n—")

        with gr.Row():
            kpi4_dash = gr.Markdown("### 🚀 Growth Score\n—")
            kpi5_dash = gr.Markdown("### 🌟 Performance Score\n—")

        gr.Markdown("---fortunate")
        gr.Markdown("## 📈 Performance Visualizations")

        with gr.Row():
            chart1_dash = gr.Plot(label="Top Products by Revenue")
            chart2_dash = gr.Plot(label="Performance Scores")

        with gr.Row():
            chart3_dash = gr.Plot(label="Sales vs Margin Analysis")
            chart4_dash = gr.Plot(label="Sales Forecast")

        back6_btn = gr.Button("⬅ Back to Data Upload", variant="secondary", size="lg")

    # ==================== EVENT HANDLERS ====================

    def update_visibility(step):
        return [
            gr.update(visible=(step == 0)), # step0_col
            gr.update(visible=(step == 1)), # step1_col
            gr.update(visible=(step == 2)), # step2_col
            gr.update(visible=(step == 3)), # step3_col
            gr.update(visible=(step == 4)), # step4_col
            gr.update(visible=(step == 5)), # step5_col
            gr.update(visible=(step == 6))  # step6_col
        ]

    def show_signup():
        return (1, *update_visibility(1))

    def handle_login(mobile):
        profile = get_user_profile(mobile)
        if profile:
            welcome_message = f"✅ Welcome back, {profile['full_name']}! You have been navigated to upload business data file for Analysis."
            # Successful login from landing page goes directly to step 5
            return (
                gr.update(value="", visible=False), # Clear landing_login_error_msg
                profile,
                5, # Set step_state to 5
                *update_visibility(5), # Updates visibility for step0-step6
                gr.update(value="", visible=False), # error1
                gr.update(value="", visible=False), # error2
                gr.update(value="", visible=False), # error3
                gr.update(value="", visible=False), # error4
                gr.update(value="", visible=False), # error5
                gr.update(value=welcome_message, visible=True) # login_welcome_message
            )
        else:
            # Failed login, stay on landing page (step 0) and show error
            return (
                gr.update(value="❌ No account found. Please register or try again.", visible=True), # Show landing_login_error_msg
                {},
                0, # Stay on step 0
                *update_visibility(0), # Updates visibility for step0-step6
                gr.update(value="", visible=False), # error1
                gr.update(value="", visible=False), # error2
                gr.update(value="", visible=False), # error3
                gr.update(value="", visible=False), # error4
                gr.update(value="", visible=False), # error5
                gr.update(value="", visible=False)  # login_welcome_message
            )

    def back_to_landing():
        return (0, *update_visibility(0))

    def validate_step1(name, mobile, email, role, current_data):
        if not name or not mobile or not role:
            return ("⚠️ Please fill all required fields", current_data, 1, *update_visibility(1), gr.update(visible=False))

        updated_data = {
            **current_data,
            'full_name': name,
            'mobile_number': mobile,
            'email': email,
            'role': role
        }
        return ("", updated_data, 2, *update_visibility(2), gr.update(visible=False))

    def verify_step2(msme_num, otp, current_data, ent_name, org, activity, ent_type, state, city, status):
        if not msme_num or not otp:
            return ("⚠️ Please fill MSME number and OTP", current_data, 2, *update_visibility(2), "", "", "", "", "", "", gr.update(visible=False))

        if otp != "1234":
            return ("⚠️ Invalid OTP", current_data, 2, *update_visibility(2), "", "", "", "", "", "", gr.update(visible=False))

        if "Successfully" not in status:
            return ("⚠️ Please fetch MSME data first", current_data, 2, *update_visibility(2), "", "", "", "", "", "", gr.update(visible=False))

        updated_data = {
            **current_data,
            'msme_number': msme_num,
            'company_name': ent_name,
            'organisation_type': org,
            'major_activity': activity,
            'enterprise_type': ent_type,
            'state': state,
            'city': city
        }

        return (
            "✅ OTP Verified",
            updated_data,
            3,
            *update_visibility(3),
            ent_name, org, activity, ent_type, state, city,
            gr.update(visible=False)
        )

    def confirm_step3(current_data, c1, c2, cert):
        if not c1 or not c2:
            return ("⚠️ Please accept both consents", current_data, 3, *update_visibility(3), gr.update(value="", visible=False), gr.update(visible=False))
        if cert is None:
            return ("⚠️ Please upload certificate", current_data, 3, *update_visibility(3), gr.update(value="", visible=False), gr.update(visible=False))

        updated_data = {**current_data, 'verification_status': 'APPROVED'}
        success_msg = f"""## ✅ Verification Status: APPROVED\n\nYour MSME certificate has been verified and approved successfully!\n\n**Company:** {current_data.get('company_name', 'N/A')}\n**MSME Number:** {current_data.get('msme_number', 'N/A')}\n**Status:** APPROVED ✓\n\nProceeding to Business Profile...\n"""
        return (gr.update(value="", visible=False), updated_data, 4, *update_visibility(4), gr.update(value=success_msg, visible=True), gr.update(visible=False))

    def submit_profile(biz_type, years, revenue, current_data):
        if not biz_type or biz_type == "Choose Business Type":
            return ("⚠️ Please select business type", current_data, gr.update(visible=False), gr.update(visible=True))
        if not revenue:
            return ("⚠️ Please select revenue range", current_data, gr.update(visible=False), gr.update(visible=True))
        if years is None or years <= 0:
            return ("⚠️ Please enter valid years in operation", current_data, gr.update(visible=False), gr.update(visible=True))

        updated_data = {
            **current_data,
            'business_type': biz_type,
            'years_operation': int(years),
            'monthly_revenue_range': revenue,
            'consent_given': True
        }

        try:
            user_id = save_user_profile(updated_data)
            success_msg = f"""## ✅ Business Profile Submitted Successfully!\n\n**Company:** {updated_data.get('company_name', 'N/A')}\n**Business Type:** {biz_type}\n**Years in Operation:** {int(years)}\n**Monthly Revenue:** {revenue}\n\nProfile saved to database (ID: {user_id})\n\nYou can now proceed to upload your business data for AI analysis.\n"""
            return (success_msg, updated_data, gr.update(visible=True), gr.update(visible=False))
        except Exception as e:
            return (f"❌ Error saving profile: {str(e)}", current_data, gr.update(visible=False), gr.update(visible=True))

    def analyze_data(user_data, consent, file):
        initial_output_updates = [
            gr.update(value="", visible=False), # insights_output
            gr.update(value=None, visible=False), # pdf_output
            gr.update(visible=False), # view_dashboard_btn
            gr.update(value="### 💰 Total Sales\n—"), # kpi1
            gr.update(value="### 📈 Total Profit\n—"), # kpi2
            gr.update(value="### 🧠 Health Score\n—"), # kpi3
            gr.update(value="### 🚀 Growth Score\n—"), # kpi4
            gr.update(value="### 🌟 Performance Score\n—"), # kpi5
            None, None, None, None, # chart1, chart2, chart3, chart4
            gr.update(value="", visible=False), # upload_message
            gr.State({}, visible=False) # dashboard_data_state component needs to be a State
        ]

        if not consent:
            return (gr.update(value="", visible=True), f"⚠️ Please provide consent to analyze data", *initial_output_updates[1:])
        if file is None:
            return (gr.update(value="", visible=True), f"⚠️ Please upload an Excel or CSV file", *initial_output_updates[1:])

        try:
            if file.name.endswith('.xlsx'):
                df = pd.read_excel(file.name)
            elif file.name.endswith('.csv'):
                df = pd.read_csv(file.name)
            else:
                return (gr.update(value="", visible=True), f"❌ Unsupported file format. Please upload .xlsx or .csv", *initial_output_updates[1:])

            required_cols = ['Date', 'Store_ID', 'Monthly_Sales_INR', 'SKU_Name', 'Monthly_Operating_Cost_INR',
                             'Outstanding_Loan_INR', 'Vendor_Delivery_Reliability', 'Inventory_Turnover',
                             'Avg_Margin_Percent', 'Monthly_Demand_Units', 'Returns_Percentage']

            for col in required_cols:
                if col not in df.columns:
                    if col == 'Date':
                        df[col] = pd.to_datetime(datetime.datetime.now().date())
                    elif col == 'SKU_Name' or col == 'Store_ID':
                        df[col] = 'Default'
                    else:
                        df[col] = 0

            insights, error_msg, forecast_data = generate_insights(user_data, df)
            if error_msg:
                return (gr.update(value="", visible=True), f"❌ {error_msg}", *initial_output_updates[1:])

            kpi1_val, kpi2_val, kpi3_val, kpi4_val, kpi5_val, fig1, fig2, fig3, fig4, dash_error = generate_dashboard_data(user_data, df)

            # Store dashboard data in state
            dashboard_data_dict = {
                'kpi1': kpi1_val,
                'kpi2': kpi2_val,
                'kpi3': kpi3_val,
                'kpi4': kpi4_val,
                'kpi5': kpi5_val,
                'chart1': fig1,
                'chart2': fig2,
                'chart3': fig3,
                'chart4': fig4
            }

            # Generate comprehensive PDF report
            pdf_path, pdf_error = generate_pdf_report(user_data, df, [fig1, fig2, fig3, fig4])

            # Create success message for insights with PDF info
            pdf_success_msg = ""
            if pdf_path and not pdf_error:
                pdf_success_msg = f"\n\n---\n\n## 📄 Comprehensive Business Report Generated!\n\n✅ Your personalized business intelligence report is ready for download!\n\n**What's in your report:**\n- Executive Summary with key insights\n- Revenue opportunities and growth strategies\n- Marketing and sales action plans\n- Operational improvement recommendations\n- Financial health analysis\n- 90-day action plan\n- Detailed charts and analytics\n\n👇 **Download your report below to access your complete business roadmap!**"

            final_insights = (insights if insights else "✅ Analysis completed successfully") + pdf_success_msg

            return (
                final_insights,
                pdf_path if pdf_path else None,
                gr.update(visible=True),
                kpi1_val, kpi2_val, kpi3_val, kpi4_val, kpi5_val,
                fig1, fig2, fig3, fig4,
                gr.update(value="", visible=False),
                dashboard_data_dict # Return the dict to update the gr.State component
            )

        except Exception as e:
            import traceback
            error_trace = traceback.format_exc()
            return (
                f"❌ Analysis failed: {str(e)}\n\n{traceback.format_exc()}\n\nPlease check your file format and ensure it contains the necessary data for analysis.",
                gr.update(value="", visible=True),
                *initial_output_updates[1:]
            )

    def show_dashboard(dashboard_data):
        """Display the dashboard with stored data"""
        # The update_visibility function returns 7 items corresponding to step0_col to step6_col
        return (
            6,
            *update_visibility(6),
            dashboard_data.get('kpi1', "### 💰 Total Sales\n—"),
            dashboard_data.get('kpi2', "### 📈 Total Profit\n—"),
            dashboard_data.get('kpi3', "### 🧠 Health Score\n—"),
            dashboard_data.get('kpi4', "### 🚀 Growth Score\n—"),
            dashboard_data.get('kpi5', "### 🌟 Performance Score\n—"),
            dashboard_data.get('chart1', None),
            dashboard_data.get('chart2', None),
            dashboard_data.get('chart3', None),
            dashboard_data.get('chart4', None)
        )

    def handle_file_upload_change(user_data, file):
        if file is not None:
            user_name = user_data.get('full_name', 'User')
            msg = f"Thank you, {user_name}, for uploading the dataset file. Click on 'Analyze Data' to view AI Insights, Dashboard, and PDF Report."
            return gr.update(value=msg, visible=True), gr.update(value="", visible=False)
        else:
            return gr.update(value="", visible=False), gr.update(value="", visible=False)

    # Wire up events
    # show_login_trigger is no longer used, as login now directly from quick_login_btn
    show_signup_trigger.click(show_signup, [], [step_state, step0_col, step1_col, step2_col, step3_col, step4_col, step5_col, step6_col])

    quick_login_btn.click(
        handle_login,
        [quick_login_mobile],
        [
            landing_login_error_msg, # New:: displays error/success on landing page
            user_data_state,
            step_state,
            step0_col, step1_col, step2_col, step3_col, step4_col, step5_col, step6_col, # Visibility outputs from update_visibility
            error1, error2, error3, error4, error5, # Clear all error messages from other steps
            login_welcome_message # Welcome message on step 5
        ]
    )

    quick_signup_btn.click(show_signup, [], [step_state, step0_col, step1_col, step2_col, step3_col, step4_col, step5_col, step6_col])

    # login_btn and back_to_landing_btn are removed with login_col

    cancel1_btn.click(lambda: (0, *update_visibility(0)), [], [step_state, step0_col, step1_col, step2_col, step3_col, step4_col, step5_col, step6_col])
    next1_btn.click(validate_step1, [name_input, mobile_input, email_input, role_input, user_data_state], [error1, user_data_state, step_state, step0_col, step1_col, step2_col, step3_col, step4_col, step5_col, step6_col])

    fetch_btn.click(_fetch_msme_data, [msme_number_input], [fetched_name, fetched_org, fetched_activity, fetched_type, fetched_state, fetched_city, fetch_status])
    back2_btn.click(lambda: (1, *update_visibility(1)), [], [step_state, step0_col, step1_col, step2_col, step3_col, step4_col, step5_col, step6_col])
    next2_btn.click(verify_step2, [msme_number_input, otp_input, user_data_state, fetched_name, fetched_org, fetched_activity, fetched_type, fetched_state, fetched_city, fetch_status],
                   [error2, user_data_state, step_state, step0_col, step1_col, step2_col, step3_col, step4_col, step5_col, step6_col, confirm_name, confirm_org, confirm_activity, confirm_type, confirm_state, confirm_city])

    back3_btn.click(lambda: (2, *update_visibility(2)), [], [step_state, step0_col, step1_col, step2_col, step3_col, step4_col, step5_col, step6_col])
    next3_btn.click(confirm_step3, [user_data_state, consent1, consent2, certificate_upload], [error3, user_data_state, step_state, step0_col, step1_col, step2_col, step3_col, step4_col, step5_col, step6_col, verification_status_display, proceed_to_step5_btn])

    back4_btn.click(lambda: (3, *update_visibility(3), gr.update(visible=True)), [], [step_state, step0_col, step1_col, step2_col, step3_col, step4_col, step5_col, step6_col, next4_btn])
    next4_btn.click(submit_profile, [business_type_input, years_input, revenue_input, user_data_state], [error4, user_data_state, proceed_to_step5_btn, next4_btn])

    proceed_to_step5_btn.click(
        lambda: (5, *update_visibility(5), gr.update(value="", visible=False)),
        [],
        [step_state, step0_col, step1_col, step2_col, step3_col, step4_col, step5_col, step6_col, error4, login_welcome_message]
    )

    back5_btn.click(lambda: (4, *update_visibility(4), gr.update(value="", visible=False)), [], [step_state, step0_col, step1_col, step2_col, step3_col, step4_col, step5_col, step6_col, login_welcome_message])
    cancel5_btn.click(lambda: (0, *update_visibility(0), gr.update(value="", visible=False)), [], [step_state, step0_col, step1_col, step2_col, step3_col, step4_col, step5_col, step6_col, login_welcome_message])
    analyze_btn.click(analyze_data, [user_data_state, consent_check, file_upload], [insights_output, pdf_output, view_dashboard_btn, kpi1, kpi2, kpi3, kpi4, kpi5, chart1, chart2, chart3, chart4, upload_message, dashboard_data_state])
    file_upload.change(handle_file_upload_change, inputs=[user_data_state, file_upload], outputs=[upload_message, error5])

    view_dashboard_btn.click(show_dashboard, [dashboard_data_state], [step_state, step0_col, step1_col, step2_col, step3_col, step4_col, step5_col, step6_col, kpi1_dash, kpi2_dash, kpi3_dash, kpi4_dash, kpi5_dash, chart1_dash, chart2_dash, chart3_dash, chart4_dash])
    back6_btn.click(lambda: (5, *update_visibility(5)), [], [step_state, step0_col, step1_col, step2_col, step3_col, step4_col, step5_col, step6_col])

if __name__ == "__main__":
    print("=" * 60)
    print("🚀 DataNetra.ai - MSME Intelligence Platform")
    print("=" * 60)
    print("✅ New Landing Page Design")
    print("✅ Header with Logo & Navigation")
    print("✅ Platform Capabilities Section")
    print("✅ Footer with LinkedIn")
    print("✅ Login/Signup Flow")
    print("=" * 60)
    demo.launch(share=True)

🚀 DataNetra.ai - MSME Intelligence Platform
✅ New Landing Page Design
✅ Header with Logo & Navigation
✅ Platform Capabilities Section
✅ Footer with LinkedIn
✅ Login/Signup Flow
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://4d20757b2c27ca7262.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
